# Schema di Corruzione Dataset

In [13]:
import pandas as pd
import numpy as np
import os

# Configurazione dei dataset da processare
DATASETS = [
    {
        'name': 'heloc_ML',
        'test_file': '../../data/processed/Fase2/SplitDataset/Split_ML/heloc_ML_imputation_test.csv',
        'out_dir': '../../data/processed/Fase2/DataCorruption/heloc_ML'
    },
    {
        'name': 'heloc_DLLM',
        'test_file': '../../data/processed/Fase2/SplitDataset/Split_DLLM/heloc_DLLM_imputation_test.csv',
        'out_dir': '../../data/processed/Fase2/DataCorruption/heloc_DLLM'
    }
]

TARGET_COL = 'RiskPerformance'
STRATEGIES = ['MCAR', 'MAR', 'MNAR']
PERCENTAGES = [0.10, 0.25, 0.40]
SEED = 42



In [14]:
def corrupt_mcar(df, p, seed):
    np.random.seed(seed)
    mask = pd.DataFrame(np.random.rand(*df.shape) < p, columns=df.columns, index=df.index)
    return mask

def corrupt_mar(df, p, seed):
    np.random.seed(seed)
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)
    
    col_info = {}
    for col in df.columns:
        try:
            q = df[col].median()
            high_mask = df[col] > q
        except TypeError:
            freq = df[col].value_counts()
            median_freq = freq.median()
            rare_cats = freq[freq <= median_freq].index
            high_mask = df[col].isin(rare_cats)
        col_info[col] = high_mask

    for col in df.columns:
        other_cols = [c for c in df.columns if c != col]
        other_col = np.random.choice(other_cols)
        
        high_mask = col_info[other_col]
            
        p_high = min(p * 1.5, 1.0)
        mean_high = high_mask.mean()
        mean_low = 1 - mean_high
        
        if mean_low > 0:
            p_low = (p - p_high * mean_high) / mean_low
            p_low = np.clip(p_low, 0.0, 1.0)
        else:
            p_low = p
            
        rand_vals = np.random.rand(len(df))
        col_mask = np.where(high_mask, rand_vals < p_high, rand_vals < p_low)
        mask[col] = col_mask
    return mask

def corrupt_mnar(df, p, seed):
    np.random.seed(seed)
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)
    for col in df.columns:
        try:
            q = df[col].median()
            high_mask = df[col] > q
        except TypeError:
            freq = df[col].value_counts()
            median_freq = freq.median()
            rare_cats = freq[freq <= median_freq].index
            high_mask = df[col].isin(rare_cats)
            
        p_high = min(p * 1.5, 1.0)
        mean_high = high_mask.mean()
        mean_low = 1 - mean_high
        
        if mean_low > 0:
            p_low = (p - p_high * mean_high) / mean_low
            p_low = np.clip(p_low, 0.0, 1.0)
        else:
            p_low = p
            
        rand_vals = np.random.rand(len(df))
        col_mask = np.where(high_mask, rand_vals < p_high, rand_vals < p_low)
        mask[col] = col_mask
    return mask

def apply_corruption(df, strategy, p, seed):
    if strategy == 'MCAR':
        return corrupt_mcar(df, p, seed)
    elif strategy == 'MAR':
        return corrupt_mar(df, p, seed)
    elif strategy == 'MNAR':
        return corrupt_mnar(df, p, seed)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")



In [15]:
# Esecuzione del processo di corruzione SOLO sull'Imputation Test (30%)

logs = []

for cfg in DATASETS:
    name = cfg['name']
    test_file = cfg['test_file']
    out_dir = cfg['out_dir']
    
    os.makedirs(out_dir, exist_ok=True)
    
    df_test = pd.read_csv(test_file, dtype=str)
    
    for c in df_test.columns:
        if c != TARGET_COL:
            try:
                df_test[c] = pd.to_numeric(df_test[c])
            except ValueError:
                pass
    
    for strategy in STRATEGIES:
        for p in PERCENTAGES:
            pct_str = f"{int(p*100)}"
            print(f"Processando {name} | {strategy} {pct_str}%")
            
            X_test = df_test.drop(columns=[TARGET_COL])
            
            current_seed = SEED + hash(f"{name}_{strategy}_{p}") % 10000
            mask_test = apply_corruption(X_test, strategy, p, current_seed)
            
            mask_test[TARGET_COL] = False
            mask_test = mask_test[df_test.columns]
            
            df_test_corrupted = df_test.copy()
            df_test_corrupted = df_test_corrupted.mask(mask_test)
            
            # Salvataggio
            test_corr_path = os.path.join(out_dir, f"{name}_imputation_test_corrupted_{strategy}_{pct_str}.csv")
            test_mask_path = os.path.join(out_dir, f"{name}_imputation_test_mask_{strategy}_{pct_str}.csv")
            
            df_test_corrupted.to_csv(test_corr_path, index=False)
            mask_test.astype(int).to_csv(test_mask_path, index=False)
            
            # Calcolo metriche log
            mask_features_only = mask_test.drop(columns=[TARGET_COL])
            n_celle_oscurate = int(mask_features_only.sum().sum())
            n_celle_totali = int(mask_features_only.size)
            missing_effettivo = n_celle_oscurate / n_celle_totali if n_celle_totali > 0 else 0.0
            
            logs.append({
                "dataset": name,
                "strategia": strategy,
                "percentuale": p,
                "seed": current_seed,
                "sottoinsieme": "Imputation Test",
                "n_celle_oscurate": n_celle_oscurate,
                "n_celle_totali": n_celle_totali,
                "missing_effettivo": missing_effettivo
            })

log_df = pd.DataFrame(logs)
log_path = "../../data/processed/Fase2/DataCorruption/run_log.csv"
os.makedirs(os.path.dirname(log_path), exist_ok=True)
log_df.to_csv(log_path, index=False)
print("Corruzione completata. Log salvato in:", log_path)
display(log_df)



Processando heloc_ML | MCAR 10%
Processando heloc_ML | MCAR 25%
Processando heloc_ML | MCAR 40%
Processando heloc_ML | MAR 10%
Processando heloc_ML | MAR 25%
Processando heloc_ML | MAR 40%
Processando heloc_ML | MNAR 10%
Processando heloc_ML | MNAR 25%
Processando heloc_ML | MNAR 40%
Processando heloc_DLLM | MCAR 10%
Processando heloc_DLLM | MCAR 25%
Processando heloc_DLLM | MCAR 40%
Processando heloc_DLLM | MAR 10%
Processando heloc_DLLM | MAR 25%
Processando heloc_DLLM | MAR 40%
Processando heloc_DLLM | MNAR 10%
Processando heloc_DLLM | MNAR 25%
Processando heloc_DLLM | MNAR 40%
Corruzione completata. Log salvato in: ../../data/processed/Fase2/DataCorruption/run_log.csv


,dataset,strategia,percentuale,seed,sottoinsieme,n_celle_oscurate,n_celle_totali,missing_effettivo
0,heloc_ML,MCAR,0.10,8647,Imputation Test,9977,100572,0.099203
1,heloc_ML,MCAR,0.25,1561,Imputation Test,24913,100572,0.247713
2,heloc_ML,MCAR,0.40,6259,Imputation Test,40049,100572,0.398212
3,heloc_ML,MAR,0.10,3657,Imputation Test,10149,100572,0.100913
4,heloc_ML,MAR,0.25,1198,Imputation Test,25070,100572,0.249274
5,heloc_ML,MAR,0.40,183,Imputation Test,40236,100572,0.400072
6,heloc_ML,MNAR,0.10,190,Imputation Test,10046,100572,0.099889
7,heloc_ML,MNAR,0.25,348,Imputation Test,25100,100572,0.249572
8,heloc_ML,MNAR,0.40,7691,Imputation Test,40018,100572,0.397904
9,heloc_DLLM,MCAR,0.10,4083,Imputation Test,6932,68034,0.101890
